In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
import warnings
from xarray.coding.times import SerializationWarning
import cftime
from scipy.optimize import curve_fit
import importlib

# my own functions
import Functions
from Functions import snow_fraction
from Functions import yearly_mean 

importlib.reload(Functions)

<module 'Functions' from '/nobackup/users/hartevel/data/Data_analysis/Functions.py'>

## Tijd over? Optimize this script so it works with libraries and uses the import functions from Functions!!

### Opening and slicing data

#### Retreiving filepaths

In [47]:
EC_earth_dir_monthly = '/nobackup/users/hartevel/data/EC_earth/data'
UKESM_dir_monthly = '/nobackup/users/hartevel/data/UKESM/data/ukesm'
CNRM_dir_monthly = '/nobackup/users/hartevel/data/CNRM/CNRM'

# data_pi = EC_earth_dir_monthly + '/pi_control_smhi'
# data_2K = EC_earth_dir_monthly + '/gwl2p0_knmi'
# data_4K = EC_earth_dir_monthly + '/gwl4p0_dmi'

# data_pi = UKESM_dir_monthly + '/pi_control_smhi'
# data_2K = UKESM_dir_monthly + '/gwl2p0_knmi'
# data_4K = UKESM_dir_monthly + '/gwl4p0_dmi'

data_pi = CNRM_dir_monthly + '/PI'
data_2K = CNRM_dir_monthly + '/GWL2'
data_4K = CNRM_dir_monthly + '/GWL4'

#nc_files = [f for f in os.listdir(data_4K) if f.endswith('.nc')]
#print(nc_files)

def files(data_scenario):
    file_tas = [f for f in os.listdir(data_scenario) if f.startswith('tas')][0]
    file_prsn = [f for f in os.listdir(data_scenario) if f.startswith('prsn')][0]
    file_pr = [f for f in os.listdir(data_scenario) if f.startswith('pr_')][0]
    return file_tas, file_prsn, file_pr
    
def filepath(data_scenario):
    filepaths = []
    for file in files(data_scenario):
        filepaths.append(data_scenario + '/'+ file)
    return filepaths

# All filepaths 
file_tas_pi, file_prsn_pi, file_pr_pi = filepath(data_pi)
file_tas_2K, file_prsn_2K, file_pr_2K = filepath(data_2K)
file_tas_4K, file_prsn_4K, file_pr_4K = filepath(data_4K)

# tas  = xr.open_dataset(file_tas_2K)['tas']
# pr   = xr.open_dataset(file_pr_2K)['pr']
# prsn = xr.open_dataset(file_prsn_2K)['prsn']
# print(tas.time, pr.time, prsn.time)


#### Opening and slicing data

In [48]:
warnings.simplefilter("ignore", SerializationWarning) # suppresses the warning that datetime does not work

def drop_duplicate_time(da, name="", verbose=True):
    """
    Remove duplicated time coordinates from a DataArray.
    Keeps the FIRST occurrence.
    """
    time = da.time

    # Boolean mask: True where time is NOT duplicated
    _, index = np.unique(time.values, return_index=True)
    index = np.sort(index)

    if verbose and len(index) < len(time):
        print(
            f"[INFO] Dropped {len(time) - len(index)} duplicate time steps from {name}"
        )

    return da.isel(time=index)


def load_and_align(tas_file, pr_file, prsn_file, verbose=True):
    tas  = xr.open_dataset(tas_file)['tas']
    pr   = xr.open_dataset(pr_file)['pr']
    prsn = xr.open_dataset(prsn_file)['prsn']

    # Remove duplicated time steps (before alignment!)
    tas  = drop_duplicate_time(tas,  name="tas",  verbose=verbose)
    pr   = drop_duplicate_time(pr,   name="pr",   verbose=verbose)
    prsn = drop_duplicate_time(prsn, name="prsn", verbose=verbose)

    # Align: keep only times present in ALL variables
    tas, pr, prsn = xr.align(tas, pr, prsn, join="inner")

    return tas, pr, prsn


tas_pi,  pr_pi,  prsn_pi  = load_and_align(file_tas_pi,  file_pr_pi,  file_prsn_pi)
tas_2K,  pr_2K,  prsn_2K  = load_and_align(file_tas_2K,  file_pr_2K,  file_prsn_2K)
tas_4K,  pr_4K,  prsn_4K  = load_and_align(file_tas_4K,  file_pr_4K,  file_prsn_4K)


# print(tas_2K.time, prsn_2K.time, pr_2K.time)

[INFO] Dropped 12 duplicate time steps from tas
[INFO] Dropped 12 duplicate time steps from pr
[INFO] Dropped 12 duplicate time steps from prsn
[INFO] Dropped 12 duplicate time steps from tas
[INFO] Dropped 12 duplicate time steps from pr
[INFO] Dropped 12 duplicate time steps from prsn
[INFO] Dropped 12 duplicate time steps from tas
[INFO] Dropped 12 duplicate time steps from pr
[INFO] Dropped 12 duplicate time steps from prsn


In [49]:

data_len = min(len(tas_pi.time), 
               len(tas_2K.time), 
               len(tas_4K.time)) 

print(data_len)

# opening PI
# These xarrays use cftime instead of Datetime
tas_pi = tas_pi.isel(time = slice(0,data_len))
pr_pi = pr_pi.isel(time = slice(0,data_len))
pr_pi = pr_pi.where(pr_pi > 1e-3, 0) 
prsn_pi = prsn_pi.isel(time = slice(0,data_len))
prsn_pi = prsn_pi.where(prsn_pi > 1e-3, 0)  # Only for CNRM

# opeing + 2K
tas_2K = tas_2K.isel(time = slice(0,data_len))
pr_2K = pr_2K.isel(time = slice(0,data_len))
pr_2K = pr_2K.where(pr_2K > 1e-3, 0)
prsn_2K = prsn_2K.isel(time = slice(0,data_len))
prsn_2K = prsn_2K.where(prsn_2K > 1e-3, 0)   # Only for CNRM

# opening + 4K
# These xarrays use cftime instead of Datetime
tas_4K = tas_4K.isel(time = slice(0,data_len))
pr_4K = pr_4K.isel(time = slice(0,data_len))
pr_4K = pr_4K.where(pr_4K > 1e-3, 0)
prsn_4K = prsn_4K.isel(time = slice(0,data_len))
prsn_4K = prsn_4K.where(prsn_4K > 1e-3, 0)   # Only for CNRM



2604


### Data processing functions

#### Removing seasonal cycle

In [50]:
def Remove_SeasCyc(data_array , plot = 0):
    t0 = data_array.time.values[0]
    new_data = []
    for month in range(1,13):
        month_data = data_array.sel(time=data_array['time'].dt.month ==month)
        month_mean = month_data.mean(dim = 'time')
        new_data.append(month_data-month_mean)

    if (data_array.time.dt.year[0] - data_array.time.dt.year[1]) == 0:
        year_mean = yearly_mean(data_array)
        mean = year_mean.mean(dim = 'time')
    else:
        mean = data_array.mean(dim = 'time')

    result = xr.concat(new_data, dim = 'time').sortby('time') + mean

    if plot == 1:
        plt.figure()
        
        if isinstance(t0, cftime.datetime):
            da = result.isel(lat=70, lon=330)
            da_original = data_array.isel(lat=70, lon=330)
            t = cftime.date2num(da.time.values, units="days since 0001-01-01")    
            plt.scatter(t, da.values, s=3, edgecolors = 'none', label = 'Seasonal cycle removed', c = 'm',)
            plt.scatter(t, da_original.values, s=3, edgecolors = 'none', label = 'Original data', c = 'black')          
        else:
            result.isel(lat = 70, lon = 330).plot.scatter(c = 'm', s = 3, edgecolors = 'none', label = 'Seasonal cycle removed')
            data_array.isel(lat = 70, lon = 330).plot.scatter(c = 'black', s = 3, edgecolors = 'none', label = 'original data')
    
        plt.legend()
        plt.title('Removal of seasonal cycle')
        plt.savefig('No_seasons_data_array.png')
        plt.show()

    return result

In [51]:
# Remove_SeasCyc(pr_pi, 1)

#### Detrending data

In [52]:
def detrend(data_array, plot = 0, lats = 20, lons = 3):
    t0 = data_array.time.values[0]
    fit_params = data_array.polyfit(dim = 'time', deg = 1)
    fit = xr.polyval(data_array.time, fit_params.polyfit_coefficients)
    difference = data_array - fit
    
    if (fit.time.dt.year[0] - fit.time.dt.year[1]) == 0:
        year_mean = yearly_mean(fit)
        mean = year_mean.mean(dim = 'time')
    else:
        mean = fit.mean(dim = 'time')
        
    detrended_array = difference + mean
   
    if plot == 1:
        plt.figure()
        
        if isinstance(t0, cftime.datetime):
            t = cftime.date2num(detrended_array.time.values, units="days since 0001-01-01")    
            # plt.scatter(t, detrended_array.isel(lat = lats, lon = lons).values, c = 'm', s = 3, edgecolors = 'none', label = 'detrended data')
            plt.scatter(t, data_array.isel(lat = lats, lon = lons).values, c = 'black', s = 3, edgecolors = 'none', label = 'original data') 
            # plt.scatter(t, difference.isel(lat = lats, lon = lons).values, c = 'b', s = 3, edgecolors = 'none', label = 'difference original data and fit')
            plt.plot(t, fit.isel(lat = lats, lon = lons).values, c = 'red', label = 'fit')  
        else:
            # detrended_array.isel(lat = lats, lon = lons).plot.scatter(x = 'time', c = 'm', s = 3, edgecolors = 'none', label = 'detrended data')
            data_array.isel(lat = lats, lon = lons).plot.scatter(x = 'time', c = 'black', s = 3, edgecolors = 'none', label = 'original data')
            # difference.isel(lat = lats, lon = lons).plot.scatter(x = 'time', c = 'b', s = 3, edgecolors = 'none', label = 'difference original data and fit')
            fit.isel(lat = lats, lon = lons).plot(c = 'red', label = 'fit')

        plt.legend()
        plt.title('Visualisation of detrending')
        plt.savefig('detrended_data_array.png')
        plt.show()
    print(data_array.min().values, detrended_array.min().values)
    return detrended_array  


In [53]:
# detrend(pr_4K, 1)

### Obtaining snow fraction

In [54]:
snfr_pi = snow_fraction(prsn_pi, pr_pi)
snfr_2K = snow_fraction(prsn_2K, pr_2K)
snfr_4K = snow_fraction(prsn_4K, pr_4K)

-9.920206
0.0
0.0


### Data processing

#### Monthly data

In [55]:
detrended_tas_pi = detrend(tas_pi)
detrended_pr_pi = detrend(pr_pi)
# detrended_pr_pi = detrended_pr_pi.where(detrended_pr_pi >=0, 0) # make negative values 0
detrended_prsn_pi = detrend(prsn_pi)
# detrended_prsn_pi = detrended_prsn_pi.where(detrended_prsn_pi >=0, 0)
print('done with pi')

detrended_tas_2K = detrend(tas_2K)
detrended_pr_2K = detrend(pr_2K)
# detrended_pr_2K = detrended_pr_2K.where(detrended_pr_2K >= 0, 0)
detrended_prsn_2K = detrend(prsn_2K)
# detrended_prsn_2K = detrended_prsn_2K.where(detrended_prsn_2K >= 0, 0)
print('done with 2K')

detrended_tas_4K = detrend(tas_4K)
detrended_pr_4K = detrend(pr_4K)
# detrended_pr_4K = detrended_pr_4K.where(detrended_pr_4K >= 0, 0)
detrended_prsn_4K = detrend(prsn_4K)
# detrended_prsn_4K = detrended_prsn_4K.where(detrended_prsn_4K >= 0, 0)

# detrended_tas_pi = detrend(Remove_SeasCyc(tas_pi))
# detrended_pr_pi = detrend(Remove_SeasCyc(pr_pi))
# detrended_prsn_pi = detrend(Remove_SeasCyc(prsn_pi))
# print('done with pi')

# detrended_tas_2K = detrend(Remove_SeasCyc(tas_2K))
# detrended_pr_2K = detrend(Remove_SeasCyc(pr_2K))
# detrended_prsn_2K = detrend(Remove_SeasCyc(prsn_2K))
# print('done with 2K')

# detrended_tas_4K = detrend(Remove_SeasCyc(tas_4K))
# detrended_pr_4K = detrend(Remove_SeasCyc(pr_4K))
# detrended_prsn_4K = detrend(Remove_SeasCyc(prsn_4K))

215.15604 214.88166337433464
0.0 -0.4739678160625047
0.0 -0.26759272497079056
done with pi
218.18001 218.18477717474724
0.0 -0.5118811071843172
0.0 -0.1448022311513011
done with 2K
220.22571 220.37818402525835
0.0 -0.5426186626248732
0.0 -0.18730505408041043


In [56]:
detrended_snfr_pi = snow_fraction(detrended_prsn_pi, detrended_pr_pi)
detrended_snfr_2K = snow_fraction(detrended_prsn_2K, detrended_pr_2K)
detrended_snfr_4K = snow_fraction(detrended_prsn_4K, detrended_pr_4K)

-9.728357876187268
-0.21770540468325095
-0.3549778089238216


In [57]:
detrended_tas_pi.to_dataset(name="tas").to_netcdf(f'{data_pi}/detrended_tas.nc')
detrended_pr_pi.to_dataset(name="pr").to_netcdf(f'{data_pi}/detrended_pr.nc')
detrended_prsn_pi.to_dataset(name="prsn").to_netcdf(f'{data_pi}/detrended_prsn.nc') 

detrended_tas_2K.to_dataset(name="tas").to_netcdf(f'{data_2K}/detrended_tas.nc')
detrended_pr_2K.to_dataset(name="pr").to_netcdf(f'{data_2K}/detrended_pr.nc')
detrended_prsn_2K.to_dataset(name="prsn").to_netcdf(f'{data_2K}/detrended_prsn.nc')

detrended_tas_4K.to_dataset(name="tas").to_netcdf(f'{data_4K}/detrended_tas.nc')
detrended_pr_4K.to_dataset(name="pr").to_netcdf(f'{data_4K}/detrended_pr.nc')
detrended_prsn_4K.to_dataset(name="prsn").to_netcdf(f'{data_4K}/detrended_prsn.nc')

detrended_snfr_pi.to_dataset(name="snfr").to_netcdf(f'{data_pi}/detrended_snfr.nc')
detrended_snfr_2K.to_dataset(name="snfr").to_netcdf(f'{data_2K}/detrended_snfr.nc')
detrended_snfr_4K.to_dataset(name="snfr").to_netcdf(f'{data_4K}/detrended_snfr.nc')

In [58]:
tas_pi.to_dataset(name="tas").to_netcdf(f'{data_pi}/sliced_tas.nc')
pr_pi.to_dataset(name="pr").to_netcdf(f'{data_pi}/sliced_pr.nc')
prsn_pi.to_dataset(name="prsn").to_netcdf(f'{data_pi}/sliced_prsn.nc') 

tas_2K.to_dataset(name="tas").to_netcdf(f'{data_2K}/sliced_tas.nc')
pr_2K.to_dataset(name="pr").to_netcdf(f'{data_2K}/sliced_pr.nc')
prsn_2K.to_dataset(name="prsn").to_netcdf(f'{data_2K}/sliced_prsn.nc')

tas_4K.to_dataset(name="tas").to_netcdf(f'{data_4K}/sliced_tas.nc')
pr_4K.to_dataset(name="pr").to_netcdf(f'{data_4K}/sliced_pr.nc')
prsn_4K.to_dataset(name="prsn").to_netcdf(f'{data_4K}/sliced_prsn.nc')

snfr_pi.to_dataset(name="snfr").to_netcdf(f'{data_pi}/sliced_snfr.nc')
snfr_2K.to_dataset(name="snfr").to_netcdf(f'{data_2K}/sliced_snfr.nc')
snfr_4K.to_dataset(name="snfr").to_netcdf(f'{data_4K}/sliced_snfr.nc')


#### Yearly data

In [59]:
detrended_yearly_tas_pi = detrend(yearly_mean(tas_pi), 0)
detrended_yearly_pr_pi = detrend(yearly_mean(pr_pi), 0)
detrended_yearly_prsn_pi = detrend(yearly_mean(prsn_pi), 0)
print('done with pi')

detrended_yearly_tas_2K = detrend(yearly_mean(tas_2K), 0)
detrended_yearly_pr_2K = detrend(yearly_mean(pr_2K), 0)
detrended_yearly_prsn_2K = detrend(yearly_mean(prsn_2K), 0)
print('done with 2K')

detrended_yearly_tas_4K = detrend(yearly_mean(tas_4K), 0)
detrended_yearly_pr_4K = detrend(yearly_mean(pr_4K), 0)
detrended_yearly_prsn_4K = detrend(yearly_mean(prsn_4K), 0)

237.54068494822883 237.29784427686664
0.0 -0.042086611401086604
0.0 -0.09236388792650951
done with pi
240.88939836057898 240.87815985484158
0.0 -0.04752159459283878
0.0 -0.024948622131058695
done with 2K
245.12215682066207 245.20480900800257
0.0 -0.1887108178015
0.0 -0.018070415035135144


In [60]:
detrended_yearly_snfr_pi = snow_fraction(detrended_yearly_prsn_pi, detrended_yearly_pr_pi)
detrended_yearly_snfr_2K = snow_fraction(detrended_yearly_prsn_2K, detrended_yearly_pr_2K)
detrended_yearly_snfr_4K = snow_fraction(detrended_yearly_prsn_4K, detrended_yearly_pr_4K)

-1.3409118350803553
-0.006006530767369789
-0.001639167047017498


In [61]:
detrended_yearly_tas_pi.to_dataset(name="tas").to_netcdf(f'{data_pi}/detrended_yearly_tas.nc')
detrended_yearly_pr_pi.to_dataset(name="pr").to_netcdf(f'{data_pi}/detrended_yearly_pr.nc')
detrended_yearly_prsn_pi.to_dataset(name="prsn").to_netcdf(f'{data_pi}/detrended_yearly_prsn.nc') 

detrended_yearly_tas_2K.to_dataset(name="tas").to_netcdf(f'{data_2K}/detrended_yearly_tas.nc')
detrended_yearly_pr_2K.to_dataset(name="pr").to_netcdf(f'{data_2K}/detrended_yearly_pr.nc')
detrended_yearly_prsn_2K.to_dataset(name="prsn").to_netcdf(f'{data_2K}/detrended_yearly_prsn.nc')

detrended_yearly_tas_4K.to_dataset(name="tas").to_netcdf(f'{data_4K}/detrended_yearly_tas.nc')
detrended_yearly_pr_4K.to_dataset(name="pr").to_netcdf(f'{data_4K}/detrended_yearly_pr.nc')
detrended_yearly_prsn_4K.to_dataset(name="prsn").to_netcdf(f'{data_4K}/detrended_yearly_prsn.nc')

In [62]:
detrended_yearly_snfr_pi.to_dataset(name="snfr").to_netcdf(f'{data_pi}/detrended_yearly_snfr.nc')
detrended_yearly_snfr_2K.to_dataset(name="snfr").to_netcdf(f'{data_2K}/detrended_yearly_snfr.nc')
detrended_yearly_snfr_4K.to_dataset(name="snfr").to_netcdf(f'{data_4K}/detrended_yearly_snfr.nc')

#### Non detrended yearly data 

In [63]:
yearly_tas_pi = yearly_mean(tas_pi)
yearly_pr_pi = yearly_mean(pr_pi)
yearly_prsn_pi = yearly_mean(prsn_pi)
print('done with pi')

yearly_tas_2K = yearly_mean(tas_2K)
yearly_pr_2K = yearly_mean(pr_2K)
yearly_prsn_2K = yearly_mean(prsn_2K)
print('done with 2K')

yearly_tas_4K = yearly_mean(tas_4K)
yearly_pr_4K = yearly_mean(pr_4K)
yearly_prsn_4K = yearly_mean(prsn_4K)

done with pi
done with 2K


In [64]:
yearly_snfr_pi = snow_fraction(yearly_prsn_pi, yearly_pr_pi)
yearly_snfr_2K = snow_fraction(yearly_prsn_2K, yearly_pr_2K)
yearly_snfr_4K = snow_fraction(yearly_prsn_4K, yearly_pr_4K)

-1.3271811821689345
0.0
0.0


In [65]:
yearly_tas_pi.to_dataset(name="tas").to_netcdf(f'{data_pi}/yearly_tas.nc')
yearly_pr_pi.to_dataset(name="pr").to_netcdf(f'{data_pi}/yearly_pr.nc')
yearly_prsn_pi.to_dataset(name="prsn").to_netcdf(f'{data_pi}/yearly_prsn.nc') 

yearly_tas_2K.to_dataset(name="tas").to_netcdf(f'{data_2K}/yearly_tas.nc')
yearly_pr_2K.to_dataset(name="pr").to_netcdf(f'{data_2K}/yearly_pr.nc')
yearly_prsn_2K.to_dataset(name="prsn").to_netcdf(f'{data_2K}/yearly_prsn.nc')

yearly_tas_4K.to_dataset(name="tas").to_netcdf(f'{data_4K}/yearly_tas.nc')
yearly_pr_4K.to_dataset(name="pr").to_netcdf(f'{data_4K}/yearly_pr.nc')
yearly_prsn_4K.to_dataset(name="prsn").to_netcdf(f'{data_4K}/yearly_prsn.nc')

yearly_snfr_pi.to_dataset(name="snfr").to_netcdf(f'{data_pi}/yearly_snfr.nc')
yearly_snfr_2K.to_dataset(name="snfr").to_netcdf(f'{data_2K}/yearly_snfr.nc')
yearly_snfr_4K.to_dataset(name="snfr").to_netcdf(f'{data_4K}/yearly_snfr.nc')

#### Inverstigating differences Detrended and Nondetreded data

In [73]:
difference = detrended_yearly_pr_pi - detrended_yearly_prsn_pi
difference = difference.sel(lat = slice(60,90))

print(difference.min().values)

print(len(difference.where(difference < 0 ,drop = True)))

# prsn = prsn + prsn_overshot # make it so snowfall cannot be larger than precipitation


-1.3409118350803553
188


In [79]:
detrended_yearly_pr_pi.sel(lat = slice(50,90)).min() # only values for pr smaller than 0 in latitudes between 0 and 30. Not relevant for me Also does not really change when looking at 

<xarray.DataArray ()> Size: 8B
array(0.11984621)
Attributes:
    standard_name:       precipitation_flux
    long_name:           Precipitation
    units:               kg m-2 s-1
    online_operation:    average
    cell_methods:        area: time: mean
    interval_operation:  900 s
    interval_write:      1 month
    description:         at surface; includes both liquid and solid phases fr...
    history:             none
    cell_measures:       area: areacella